this notebook is for deciding on some model parameters and module development. namely, we will have to decide:

1. the embedding dimension, and positional embedding
2. the number of heads for multi-head attention, and MLP dimension
3. training schedule and parameters

in the spirit of how our other parameters have been chosen, we use GPT-2 as a benchmark for our model parameters. GPT-2 had variable embedding dimensions depending on the model size, with 768 dimensions for the smallest model. we choose the same dimensions to avoid the compute cost of figuring out the optimal embedding size. we will revisit the optimality of our decisions at a later point.

on the point of the amount of heads, this is generally backed out of each head's output dimension. a common choice is 64 (which is also the ), which would suggest 12 heads for our embedding dimension of 768; but empirical research points to the fact that the number of heads often matters much less than one would assume (Michel & Levy, 2019); so we will not ponder too much on this problem.

in terms of training schedules and parameters, this is the main empirical addition to our research in this notebook. but first, we must write modules which will allow us to train our neural network. of course, the below declarations are purely for development and documentation purposes, and standalone modules are contained in some appropiate path in the project.

### development of modules

In [2]:
import torch
import numpy as np
import torch.nn as nn

class MultiHeadAttentionDecoder(nn.Module):
    def __init__(self, n_dim: int, n_heads: int):
        super().__init__()
        if n_dim <= 0 or n_heads <= 0:
            raise ValueError('MultiHeadAttention: one or more parameters passed in is negative.')
        elif n_dim % n_heads != 0:
            raise ValueError('MultiHeadAttention: number of heads is not divisible by the embedding dimension')

        # linear projection layers to split later
        self.attn_proj = nn.Linear(n_dim, n_dim * 3)
        self.n_heads = n_heads

        # post-concat mixing of multi-head attention outputs
        self.out_proj = nn.Linear(n_dim, n_dim)

    def forward(self, x: torch.Tensor):
        x_proj: torch.Tensor = self.attn_proj(x)
        q, k, v = x_proj.chunk(3, dim=-1)

        # batches, tokens and embedding dimension
        B, T, C = q.shape
        H = self.n_heads
        D = C // H

        # split the output dimension into heads; and then transpose such that heads is the second dimension
        # this means that matrix multiplications will treat the (batch, head) as the batch dimension
        q = q.view(B, T, H, D).transpose(1, 2)
        k = k.view(B, T, H, D).transpose(1, 2)
        v = v.view(B, T, H, D).transpose(1, 2)

        # get attention scores
        attention = q @ k.transpose(-2, -1)
        attention = attention / np.sqrt(D)

        mask = torch.triu(torch.ones(T, T, dtype=torch.bool, device=x.device), diagonal=1)

        # causal mask, then soft max on keys
        attention = attention.masked_fill(mask, float('-inf'))
        attention = attention.softmax(dim=-1)

        out = attention @ v
        # remerge output, then pass through linear output layer
        out = out.transpose(1, 2).reshape(B, T, C)

        return self.out_proj(out)

we can produce a small example of the scores that the attention block should produce.

In [2]:
x = torch.tensor([
    [[ 0.1, -0.2,  0.3,  0.0,  0.5, -0.1,  0.2,  0.4],
     [ 0.7,  0.1, -0.3,  0.2,  0.0,  0.6, -0.5,  0.1],
     [-0.4,  0.3,  0.8, -0.2,  0.1,  0.0,  0.3, -0.6],
     [ 0.2,  0.9, -0.1,  0.5, -0.3,  0.2,  0.0,  0.7]]
])

mtah = MultiHeadAttentionDecoder(n_dim=8, n_heads=2)
out = mtah(x)
print(out)

tensor([[[-0.1334, -0.3179,  0.1450,  0.3589,  0.1807,  0.1820,  0.0996,
          -0.0047],
         [-0.2039, -0.2524,  0.1739,  0.4630,  0.2589,  0.2497,  0.0845,
          -0.0079],
         [-0.1405, -0.3301,  0.1311,  0.3509,  0.2766,  0.1392,  0.0310,
          -0.0524],
         [-0.1593, -0.2772,  0.1388,  0.3536,  0.2196,  0.2102,  0.0717,
           0.0018]]], grad_fn=<ViewBackward0>)


each block in a transformer contains an attention block and a feed forward/MLP block, so we can collate the two blocks into one module; i repeat imports so that these code cells are individual modules. note that our transformer blocks are pre-norm versus post-norm, which is the original implemnentation of transformers in Vaswani et. al (2017).

In [3]:
import torch
import torch.nn as nn

class TransformerBlock(nn.Module):
    def __init__(self, n_dim: int, n_heads: int, mlp_dim: int):
        super().__init__()
        if n_dim <= 0 or n_heads <= 0: raise ValueError('MultiHeadAttention: one or more parameters passed in is negative.')
        elif n_dim % n_heads != 0: raise ValueError('MultiHeadAttention: number of heads is not divisible by the embedding dimension')

        self.attn = MultiHeadAttentionDecoder(n_dim, n_heads)
        self.ln1 = nn.LayerNorm(n_dim)
        self.mlp = nn.Sequential(nn.Linear(n_dim, mlp_dim), nn.GELU(), nn.Linear(mlp_dim, n_dim))
        self.ln2 = nn.LayerNorm(n_dim)

    def forward(self, x: torch.Tensor):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

In [4]:
tb = TransformerBlock(n_dim=8, n_heads=2, mlp_dim=8*4)
out = tb(x)
print(out)

tensor([[[-0.3506,  0.3998,  0.3498,  0.1633,  0.6213, -0.1900, -0.1186,
           0.6916],
         [ 1.0330,  0.3998,  0.0037,  0.8664, -0.3036,  0.6052,  0.0079,
          -0.0566],
         [-0.3056,  0.4571,  1.0131, -0.3050, -0.1397, -0.4388, -0.1879,
          -0.4498],
         [ 0.2244,  1.2373,  0.1141,  0.5958, -0.3666, -0.0916,  0.0066,
           0.6668]]], grad_fn=<AddBackward0>)


now to create a full transformer neural net, we generally stack many of these transformer blocks in sequence. each transformer block is a self contained, residual connected block, so we can pass through the outputs of each block into another without any processing. the full net also requires some external layers that are not contained within the transformer block, namely:
- a token embedding layer
- a positional embedding layer
- an output layer, which outputs embedding dimension to vocab size, for token prediction

In [4]:
import torch
import torch.nn as nn
from typing import List

class TransformerNet(nn.Module):
    def __init__(self, vocab_size: int, n_dim: int, seq_len: int, n_heads: List[int], mlp_dim: List[int]):
        super().__init__()
        if len(n_heads) != len(mlp_dim): raise ValueError('TransformerNet: the number of values given in n_heads does not equal the number of values in mlp_dim.')

        self.seq_len    = seq_len
        # token and position embeddings. absolute positional embedding
        self.tok_emb    = nn.Embedding(vocab_size, n_dim)
        self.pos_emb    = nn.Embedding(seq_len, n_dim)
        self.blocks     = nn.Sequential(*[ TransformerBlock(n_dim, n_heads[i], mlp_dim[i]) for i in range(len(n_heads)) ])
        # final layer normalisation
        self.ln         = nn.LayerNorm(n_dim)
        # cast vector embedding dimension to vocab size for token prediction
        self.head       = nn.Linear(n_dim, vocab_size, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x will batches x tokens passed in
        _, T = x.shape
        if T > self.seq_len: raise ValueError(f'Sequence length {T} exceeds max_seq_len {self.seq_len}')
        # arange creates a range vector [0, T)
        pos = torch.arange(T, device=x.device)
        x = self.tok_emb(x) + self.pos_emb(pos)
        x = self.blocks(x)
        x = self.ln(x)
        return self.head(x)

### gauging computation limits

we choose a sequence length of `512`. we still have a few important decisions to make:
1. what is the embedding dimension?
2. how many transformer blocks do we want?
3. what dimension should our MLP layers be?

we can back out our desired numbers from our end goal of $\approx 100M$ parameters. for the embedding dimension, we again stick with GPT-2 and choose `768`. now, consider what each transformer block would contribution to the parameter count (assuming all transformer blocks are given the same parameters); given the embedding dimension $d$
- the key, query and value linear layers cost $d \cdot d$ each, plus an output layer, leading to $4d^2$
- then, the MLP layer is generally an expansion of `~4x` the embedding dimension, which would add $d \cdot 4d + 4d \cdot d = 8d^2$

thus each transformer block we have $12d^2$ parameters, or with $d = 768$ ~7.1m parameters. we will go with around ~12 blocks.

In [6]:
device = torch.device('cuda')

test_net = TransformerNet(vocab_size=20_000, n_dim=768, seq_len=512, n_heads=[12] * 12, mlp_dim=[768 * 4] * 12)
test_net.to(device);

we want to know what a reasonable number of samples is within a batch, and how long training could take, given maximum compute usage. we do this by using our implemented dataset class and data loaders and consider a mix of runtime and GPU utilisation. try batches of 2, 4, 8, 16 and 32.

In [7]:
import os
import time
from dotenv import load_dotenv
from torch.utils.data import DataLoader, default_collate
from askwiki.data.classes.token import TokenSequence

load_dotenv()
data_path = os.environ['DATAPATH']

token_data = TokenSequence(f'{data_path}/train.bin', np.uint16, 512)
def get_batch(B, start=0):
    x, y = default_collate([token_data[i] for i in range(start, start + B)])
    return x.to(device, dtype=torch.long), y.to(device, dtype=torch.long)

batches = {B: get_batch(B) for B in (2, 4, 8, 16, 32)}

we can measure the time it takes to run each batch length below.

In [8]:
def measure_perf(net: nn.Module, batch):
    s = time.perf_counter()
    test_net(batch)
    e = time.perf_counter()
    print(f'{e - s} seconds')


with torch.no_grad():
    measure_perf(test_net, batches[2][0])
    measure_perf(test_net, batches[4][0])
    measure_perf(test_net, batches[8][0])
    measure_perf(test_net, batches[16][0])
    measure_perf(test_net, batches[32][0])

0.27452209999319166 seconds
0.007287999993423 seconds
0.007919600000604987 seconds
0.008940200001234189 seconds
0.011663599987514317 seconds


so those run pretty quick, which is great; but we also need to simulate a more realistic environment for testing where there is an optimiser, as well as gradient storage which will incur memory costs.

In [9]:
import torch.nn.functional as F

def measure(net: nn.Module, x, y, train=True):
    net.zero_grad(set_to_none=True)

    # find the amount of vRAM used before the batch sizes are tested
    torch.cuda.synchronize()
    base = torch.cuda.memory_allocated()
    torch.cuda.reset_peak_memory_stats()

    # test the runtime of the batch size
    s = time.perf_counter()
    with torch.set_grad_enabled(train), torch.autocast('cuda', dtype=torch.bfloat16):
        logits = net(x)
        loss = F.cross_entropy(logits.flatten(0, 1), y.flatten())
    if train:
        loss.backward()
    torch.cuda.synchronize()
    t = time.perf_counter() - s

    # how much vRAM is the model occupying now?
    peak = torch.cuda.max_memory_allocated()
    net.zero_grad(set_to_none=True)
    print(f'B={x.size(0):>3} | {t * 1000:7.1f} ms | peak {peak / 1e9:5.2f} GB | batch adds {(peak - base) / 1e9:5.2f} GB')

# for warmup
measure(test_net, *batches[2]);
for B in (2, 4, 8, 16, 32):
    measure(test_net, *batches[B])

B=  2 |   221.5 ms | peak  1.74 GB | batch adds  1.26 GB
B=  2 |    45.7 ms | peak  1.74 GB | batch adds  1.26 GB
B=  4 |    88.2 ms | peak  2.78 GB | batch adds  2.30 GB
B=  8 |   201.9 ms | peak  4.87 GB | batch adds  4.38 GB
B= 16 |  1660.5 ms | peak  9.05 GB | batch adds  8.57 GB
B= 32 | 16045.9 ms | peak 17.39 GB | batch adds 16.91 GB


ok so the above shows us that > 8 samples per batch is not really feasible for us as we have 8GB of VRAM, so there is likely some offloading happening there. the run time of the 8 samples per batch takes roughly double the time it takes for 4 samples which gives us a clue that nothing is being offloaded. for learning rates, etc - we will make these decisions in training time, once we are able to create a training module.